# TradeSense · TGNN branch · "Walk-the-book" consistency test (read-only) → `09_tgnn_tardis_walk_the_book_v1`

**Why this notebook exists.** Notebook 08 showed that 44 % of BTCUSDT trades print ≥ 2 ticks deeper than the best bid/ask of the last book message, and that this is strongly tied to *bursts*: the first trade after a book update is at the touch 96.7 % of the time, while the 11th-and-later trades in the same update interval are deep 71.5 % of the time. The depth feed arrives in ~100 ms batches (measured median 100.05 ms), the trade feed does not. So a burst of marketable orders can walk through several price levels **inside** one book interval, and the book only shows the result at the next message.

**The question this notebook answers, without assuming the answer:** *at 100 ms resolution, are the trades consistent with the displayed book being consumed?* For every book interval and each taker side it takes the book as displayed at the start of the interval, replays the trades of that interval in order, and tests two physical conditions for each trade (the book is the full-depth reference replay):

* **Reach test.** To print at price *p*, all levels better than *p* must already have been eaten. So the volume traded before this trade (same side, same interval, at displayed prices) must be ≥ the displayed quantity strictly better than *p*.
* **Exhaustion test.** A level cannot supply more than it displayed. So the volume traded through this trade must be ≤ the displayed quantity from the touch up to and including *p*.

If both hold, the trade is exactly what "the displayed book was consumed" predicts. If the reach test fails, the price went deeper than the traded volume explains (liquidity ahead of it was cancelled or moved). If the exhaustion test fails, more volume traded than was displayed (the level was refilled/new orders arrived inside the 100 ms). Neither failure is a data error by itself — both are normal market behaviour that a 100 ms snapshot cannot show — but their *size* tells us how much of the trade tape can be tied to the book.

**Classes (mutually exclusive, per trade)**
`0 consistent` · `1 reach violated (deeper than the consumed volume explains)` · `2 exhaustion violated (more volume than displayed up to this level)` · `3 both violated` · `4 price not a displayed level` · `5 better than the touch, inside the spread` · `6 at/through the opposite touch` · `7 further than the 30,000-tick walk cap` · `8 no valid book state` · `9 unknown side`.
Trades in classes 4–6 are not counted as consuming displayed depth. Tolerances: 1e-7 absolute (+1e-9 relative) on quantities.

**Boundaries.** Reads only the files saved by notebook 07 (never touches the network, downloads nothing), never modifies them, trains nothing, builds no graphs, creates no synthetic events. Output goes to a new folder `TGNN_Tardis_WalkBook_BTCUSDT_2024-05-01_v1` (refuses to run if it exists). The earlier audit's and diagnostic's thresholds and results are untouched. Trades and book come from the same provider, so "consistent" means consistent with the provider's feed, not independently validated; Tardis' CSV has no order IDs, so nothing here identifies individual orders or traders and nothing here detects manipulation.

**Evidence rules frozen before the data is seen.** Let *W* = share of all usable trades in class 0 and *Qc* = share of the deep trades (≥ 2 ticks deeper than the pre-state touch) in class 0. Let *I* = share of all usable trades in classes 5 + 6.
* Interpretation of *Qc*: **≥ 60 %** deep trades are largely what consuming the displayed book predicts · **30–60 %** partly · **< 30 %** mostly not.
* **Recommendation (frozen):** all E and D tests must pass, otherwise **HOLD**. Then: **PROCEED** (ask for approval of the remaining free days) if *W* ≥ 70 % **and** *I* ≤ 6 %; **PROCEED FOR BOOK DATA ONLY** (trades only as per-interval aggregates, never matched price-by-price to the book) if 40 % ≤ *W* < 70 %, or *W* ≥ 70 % but *I* > 6 %; **HOLD** if *W* < 40 %. These cut-offs are judgement calls made in advance, not statistical tests. Nothing is downloaded without your approval.

**Test groups.** **E** = notebook execution tests, **D** = input integrity re-check. Passing E and D validates neither the book nor the trades.

## 1 · Setup, read-only guard, and a fingerprint of everything that already exists

In [ ]:
import os, sys, io, re, gc, json, time, math, gzip, uuid, shutil, hashlib, heapq, warnings, resource
from array import array
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.csv as pacsv
import pyarrow.compute as pc
warnings.filterwarnings('ignore')
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 100)
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
# NOTE: this notebook deliberately imports no HTTP library and never touches the network.
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets'))
assert BASE.is_dir(), f'Drive folder not found: {BASE}'
SRC = BASE / 'TGNN_Tardis_Binance_Spot_BTCUSDT_2024-05-01_v1'
OUT_FINAL = BASE / 'TGNN_Tardis_WalkBook_BTCUSDT_2024-05-01_v1'
DIAG08 = BASE / 'TGNN_Tardis_TradeDiag_BTCUSDT_2024-05-01_v1' / 'reports' / 'trade_diagnostic_report.json'     # optional: used only for a cross-notebook consistency test
STAGE = BASE / ('.tardis_walk_stage_' + uuid.uuid4().hex[:8])
assert SRC.is_dir(), f'Source folder from notebook 07 not found: {SRC}'
assert not OUT_FINAL.exists(), f'REFUSING to overwrite {OUT_FINAL}'
EXCHANGE, SYMBOL, DAY = 'binance', 'BTCUSDT', '2024-05-01'
DAY0_US = int(datetime(2024, 5, 1, tzinfo=timezone.utc).timestamp() * 1_000_000); DAY1_US = DAY0_US + 86_400_000_000
FILES = {dt: SRC / 'raw' / f'{EXCHANGE}_{dt}_{DAY}_{SYMBOL}.csv.gz' for dt in ('incremental_book_L2', 'trades', 'book_snapshot_25')}
for k in ('incremental_book_L2', 'trades'): assert FILES[k].is_file(), f'missing {FILES[k]}'
HAVE_S25 = FILES['book_snapshot_25'].is_file()
UTC = lambda: datetime.now(timezone.utc).isoformat(); RUN_STARTED = UTC(); STAGES = {}; REPORT = {}
def sha256_file(p, chunk=16 << 20):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''): h.update(b)
    return h.hexdigest()
def jsonable(o):
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return None if not np.isfinite(o) else float(o)
    if isinstance(o, np.bool_): return bool(o)
    if isinstance(o, np.ndarray): return o.tolist()
    return str(o)
def dump_json(obj, path):
    path.parent.mkdir(parents=True, exist_ok=True); path.write_text(json.dumps(obj, indent=2, default=jsonable), encoding='utf-8')
def peak_mem_gb(): return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6
def tick_(name, t0): STAGES[name] = {'seconds': round(time.time() - t0, 1), 'peak_mem_gb_so_far': round(peak_mem_gb(), 2)}; print(f'[{name}] {STAGES[name]["seconds"]} s | peak memory so far {STAGES[name]["peak_mem_gb_so_far"]} GB')
def sh(a, b): return (a / b) if b else float('nan')
t0 = time.time(); PIN0 = {str(p.relative_to(BASE)): (p.stat().st_size, p.stat().st_mtime_ns) for p in BASE.rglob('*') if p.is_file()}
print(f'fingerprinted {len(PIN0)} existing files (including the 07 output folder); they will be re-checked at the end'); tick_('fingerprint', t0)
MAN = json.loads((SRC / 'manifest.json').read_text()); SRC_REPORT = json.loads((SRC / 'reports' / 'audit_report.json').read_text()); SRC_SHARE = (SRC / 'reports' / 'share_block.txt').read_text()
print('source folder:', SRC); print('book_snapshot_25 present:', HAVE_S25)

Mounted at /content/drive
fingerprinted 768 existing files (including the 07 output folder); they will be re-checked at the end
[fingerprint] 45.3 s | peak memory so far 0.17 GB
source folder: /content/drive/MyDrive/research datasets/TGNN_Tardis_Binance_Spot_BTCUSDT_2024-05-01_v1
book_snapshot_25 present: True


## 2 · Load the saved files (read-only) and check them against the SHA-256 recorded by notebook 07

In [ ]:
t0 = time.time()
SHA_CHECK = {}
for dt, p in FILES.items():
    if not p.is_file(): continue
    key = 'raw/' + p.name; SHA_CHECK[dt] = {'file': p.name, 'recorded': MAN.get(key), 'now': sha256_file(p)}; SHA_CHECK[dt]['equal'] = SHA_CHECK[dt]['recorded'] == SHA_CHECK[dt]['now']
    print(dt, 'sha256 equal to manifest:', SHA_CHECK[dt]['equal'])
tick_('sha256_recheck', t0)
t0 = time.time()
try: tr_tbl = pacsv.read_csv(FILES['trades'], convert_options=pacsv.ConvertOptions(column_types={'exchange': pa.string(), 'symbol': pa.string(), 'timestamp': pa.int64(), 'local_timestamp': pa.int64(), 'id': pa.int64(), 'side': pa.string(), 'price': pa.float64(), 'amount': pa.float64()}))
except Exception as e: raise RuntimeError('trade ids are not plain integers -> STOP and paste this to Claude: ' + str(e)[:200])
tE = tr_tbl.column('timestamp').to_numpy(); tL = tr_tbl.column('local_timestamp').to_numpy(); tid = tr_tbl.column('id').to_numpy(); tpx = tr_tbl.column('price').to_numpy(); tam = tr_tbl.column('amount').to_numpy()
side_s = tr_tbl.column('side'); tside = np.full(len(tE), 2, np.int8); tside[pc.equal(side_s, 'buy').to_numpy(zero_copy_only=False).astype(bool)] = 0; tside[pc.equal(side_s, 'sell').to_numpy(zero_copy_only=False).astype(bool)] = 1
NT = len(tE); del tr_tbl; gc.collect(); tick_('load_trades', t0); print(f'trades loaded: {NT:,} | buy {int((tside==0).sum()):,} sell {int((tside==1).sum()):,} other {int((tside==2).sum()):,}')
TRADES_SORTED_LOCAL = bool(np.all(np.diff(tL) >= 0)); TRADES_SORTED_EXCH = bool(np.all(np.diff(tE) >= 0)); print('trades sorted by local ts:', TRADES_SORTED_LOCAL, '| by exchange ts:', TRADES_SORTED_EXCH)
if not TRADES_SORTED_LOCAL: raise RuntimeError('trades are not in local-timestamp order: STOP and paste this to Claude.')
BOOK_TYPES = {'exchange': pa.string(), 'symbol': pa.string(), 'timestamp': pa.int64(), 'local_timestamp': pa.int64(), 'is_snapshot': pa.bool_(), 'side': pa.string(), 'price': pa.float64(), 'amount': pa.float64()}
def book_reader(block_mb=48): return pacsv.open_csv(FILES['incremental_book_L2'], read_options=pacsv.ReadOptions(block_size=block_mb << 20), convert_options=pacsv.ConvertOptions(column_types=BOOK_TYPES, strings_can_be_null=False, true_values=['true', 'True', 'TRUE'], false_values=['false', 'False', 'FALSE']))
# tick size from the data itself: the LARGEST "nice" step (1, 2, 5 x 10^k) on which every price in the first book block and in the trades lies
first_blk = next(iter(book_reader(16))); pr0 = np.unique(first_blk.column('price').to_numpy())
def off_grid_share(p, tk): x = p / tk; return float(np.mean(np.abs(x - np.rint(x)) > 1e-4))
CANDS = sorted({m * 10.0 ** k for k in range(-8, 3) for m in (1, 2, 5)}, reverse=True); probe_p = np.concatenate([pr0, tpx[:200_000]])
TICK = next((c for c in CANDS if off_grid_share(probe_p, c) == 0.0), None)
if TICK is None: raise RuntimeError('no tick size fits all prices: STOP and paste this to Claude.')
TICK = float(TICK); print('tick size inferred from the data:', TICK)
GRID = {'book_first_block_off_grid_share': off_grid_share(pr0, TICK), 'trades_off_grid_share': off_grid_share(tpx, TICK)}; print(GRID)
tpt = np.rint(tpx / TICK).astype(np.int64)

incremental_book_L2 sha256 equal to manifest: True
trades sha256 equal to manifest: True
book_snapshot_25 sha256 equal to manifest: True
[sha256_recheck] 4.2 s | peak memory so far 0.2 GB
[load_trades] 1.7 s | peak memory so far 0.49 GB
trades loaded: 2,401,094 | buy 1,142,546 sell 1,258,548 other 0
trades sorted by local ts: True | by exchange ts: True
tick size inferred from the data: 0.01
{'book_first_block_off_grid_share': 0.0, 'trades_off_grid_share': 0.0}


## 3 · The walk-the-book test (one implementation, used by the replay and by the unit tests)

`walk_classify` takes the displayed quantities of one side at the start of a book interval, the touch, and the trades of that side in time order, and returns for each trade its class and the quantities used by the two tests. It walks tick by tick from the touch to the deepest traded price once per interval and side.

In [ ]:
WALK_CAP = 30_000; TOL_ABS = 1e-7; TOL_REL = 1e-9
def walk_classify(dct, touch, dirn, opp, prices, amts, cap=WALK_CAP):
    # dct: {tick: displayed qty} for the side being hit; dirn = -1 for sell-takers (deeper = lower price), +1 for buy-takers
    n = len(prices); cls = [0] * n; Db = [0.0] * n; Di = [0.0] * n; Cp = [0.0] * n; dist = [0] * n; need = set()
    for i in range(n):
        d = (prices[i] - touch) * dirn; dist[i] = d
        if d < 0: cls[i] = 6 if ((dirn < 0 and prices[i] >= opp) or (dirn > 0 and prices[i] <= opp)) else 5
        elif d > cap: cls[i] = 7
        else: need.add(d)
    res = {}; k = 0; cum = 0.0
    for dd_ in sorted(need):
        while k < dd_: cum += dct.get(touch + dirn * k, 0.0); k += 1
        pk = touch + dirn * dd_; res[dd_] = (cum, cum + dct.get(pk, 0.0), pk in dct)
    C = 0.0
    for i in range(n):
        if cls[i] in (5, 6, 7): continue
        db, di, shown = res[dist[i]]; Db[i] = db; Di[i] = di; Cp[i] = C
        if not shown: cls[i] = 4; continue
        a = amts[i]; reach_ok = C >= db - TOL_ABS - TOL_REL * db; exh_ok = (C + a) <= di + TOL_ABS + TOL_REL * di
        cls[i] = 0 if (reach_ok and exh_ok) else 1 if (not reach_ok and exh_ok) else 2 if (reach_ok and not exh_ok) else 3
        C += a
    return cls, Db, Di, Cp, dist
CLS_NAMES = {0: 'consistent', 1: 'reach_violated(deeper_than_volume_explains)', 2: 'exhaustion_violated(more_than_displayed)', 3: 'both_violated', 4: 'price_not_displayed_level', 5: 'better_than_touch_inside_spread', 6: 'at_or_through_opposite_touch', 7: 'beyond_walk_cap', 8: 'no_valid_state', 9: 'unknown_side'}

## 4 · Full-depth replay with trades interleaved (one pass)

Same discipline as notebook 08: a trade is judged against the book after the last completed message with `local_timestamp` strictly before the trade; trades are grouped by the book interval they fall in (between two consecutive messages) and by taker side. After the next message the post-interval touch is recorded so that the notebook can say whether the touch actually moved to (or through) the deepest price traded.

In [ ]:
class IBook:
    __slots__ = ('bids', 'asks', 'bh', 'ah')
    def __init__(s): s.bids = {}; s.asks = {}; s.bh = []; s.ah = []
    def reset(s): s.bids.clear(); s.asks.clear(); s.bh.clear(); s.ah.clear()
    def upd(s, is_bid, p, a):
        if is_bid:
            d = s.bids
            if a > 0:
                if p not in d: heapq.heappush(s.bh, -p)
                d[p] = a
            else: d.pop(p, None)
        else:
            d = s.asks
            if a > 0:
                if p not in d: heapq.heappush(s.ah, p)
                d[p] = a
            else: d.pop(p, None)
    def best(s):
        bh = s.bh; bd = s.bids
        while bh and -bh[0] not in bd: heapq.heappop(bh)
        ah = s.ah; ad = s.asks
        while ah and ah[0] not in ad: heapq.heappop(ah)
        if len(bh) > 4 * len(bd) + 4000: s.bh = bh = [-p for p in bd]; heapq.heapify(bh)
        if len(ah) > 4 * len(ad) + 4000: s.ah = ah = list(ad); heapq.heapify(ah)
        return (-bh[0] if bh else 0), (ah[0] if ah else 0)
TL_ = tL.tolist(); TP_ = tpt.tolist(); TS_ = tside.tolist(); TA_ = tam.tolist()

def load_s25_best(stride):
    out = {}; n = 0
    if not HAVE_S25: return out, 0
    rd = pacsv.open_csv(FILES['book_snapshot_25'], read_options=pacsv.ReadOptions(block_size=48 << 20), convert_options=pacsv.ConvertOptions(include_columns=['local_timestamp', 'asks[0].price', 'bids[0].price'], column_types={'local_timestamp': pa.int64(), 'asks[0].price': pa.float64(), 'bids[0].price': pa.float64()}))
    for b in rd:
        L = b.column('local_timestamp').to_numpy(); A = b.column('asks[0].price').to_numpy(); B = b.column('bids[0].price').to_numpy(); idx = np.arange((-n) % stride, len(L), stride); n += len(L)
        At = np.rint(A[idx] / TICK).astype(np.int64).tolist(); Bt = np.rint(B[idx] / TICK).astype(np.int64).tolist()
        for l, a, bb in zip(L[idx].tolist(), At, Bt): out[l] = (a, bb)
    return out, n

def run_pass(max_rows=None, samp=None, verbose=True):
    rd = book_reader(); bk = IBook(); n = NT
    cls = np.full(n, 8, np.int8); dist = np.zeros(n, np.int32); Dbq = np.zeros(n); Diq = np.zeros(n); Cpq = np.zeros(n); gsz = np.zeros(n, np.int32); gvol = np.zeros(n); post_ok = np.full(n, -1, np.int8); age = np.full(n, -1, np.int64); touchq = np.zeros(n)
    ML = array('q'); ME = array('q'); MB = array('q'); MA = array('q')
    st = {'rows': 0, 'skipped_pre': 0, 'messages': 0, 'resets': 0, 'crossed': 0, 'locked': 0, 'snap_checked': 0, 'snap_best_equal': 0, 'trades_after_last_message': 0, 'groups': 0}
    S = {'j': 0, 'L': None, 'bb': 0, 'ba': 0}; pend = []
    def flush(Lcut):
        j = S['j']
        if j >= n or TL_[j] > Lcut: return
        j1 = j
        while j1 < n and TL_[j1] <= Lcut: j1 += 1
        bb = S['bb']; ba = S['ba']; cL = S['L']; valid = cL is not None and bb > 0 and ba > 0 and bb < ba
        if not valid:
            for t in range(j, j1): cls[t] = 9 if TS_[t] == 2 else 8
            S['j'] = j1; return
        sells = []; buys = []
        for t in range(j, j1):
            s = TS_[t]; age[t] = TL_[t] - cL
            if s == 1: sells.append(t)
            elif s == 0: buys.append(t)
            else: cls[t] = 9
        for lst, dct, touch, dirn, opp in ((sells, bk.bids, bb, -1, ba), (buys, bk.asks, ba, 1, bb)):
            if not lst: continue
            pr = [TP_[t] for t in lst]; am = [TA_[t] for t in lst]; c, Db, Di, Cp, ds = walk_classify(dct, touch, dirn, opp, pr, am); st['groups'] += 1; vol = sum(am); tq0 = dct.get(touch, 0.0); pw = None
            for q, t in enumerate(lst):
                cls[t] = c[q]; dist[t] = ds[q]; Dbq[t] = Db[q]; Diq[t] = Di[q]; Cpq[t] = Cp[q]; gsz[t] = len(lst); gvol[t] = vol; touchq[t] = tq0
                if ds[q] >= 0 and c[q] != 7 and (pw is None or ds[q] > pw): pw = ds[q]
            if pw is not None: pend.append((lst, dirn, touch + dirn * pw))
        S['j'] = j1
    def end_msg(L, E):
        st['messages'] += 1; bb, ba = bk.best(); ML.append(L); ME.append(E); MB.append(bb); MA.append(ba)
        if bb and ba:
            if bb > ba: st['crossed'] += 1
            elif bb == ba: st['locked'] += 1
        S['L'] = L; S['bb'] = bb; S['ba'] = ba
        if pend:
            for lst, dirn, pw in pend:
                ok = (bb > 0 and bb <= pw) if dirn < 0 else (ba > 0 and ba >= pw)
                post_ok[lst] = 1 if ok else 0
            pend.clear()
        if samp is not None and L in samp and bb and ba and bb < ba:
            a0, b0 = samp[L]; st['snap_checked'] += 1; st['snap_best_equal'] += int(a0 == ba and b0 == bb)
    started = False; pL = None; pf = None; pE = None; stop = False; t0 = time.time(); done = 0
    for b in rd:
        Ls = b.column('local_timestamp').to_numpy().tolist(); Es = b.column('timestamp').to_numpy().tolist(); Fs = b.column('is_snapshot').to_numpy(zero_copy_only=False).astype(bool).tolist()
        Bs = pc.equal(b.column('side'), 'bid').to_numpy(zero_copy_only=False).astype(bool).tolist(); Ps = np.rint(b.column('price').to_numpy() / TICK).astype(np.int64).tolist(); As = b.column('amount').to_numpy().tolist()
        for i in range(len(Ls)):
            if max_rows is not None and st['rows'] >= max_rows: stop = True; break
            L = Ls[i]; f = Fs[i]; st['rows'] += 1
            if started and (f != pf or ((not f) and L != pL)):
                end_msg(pL, pE); flush(L)
            if not started:
                if not f: st['skipped_pre'] += 1; pL = L; pf = f; pE = Es[i]; continue
                started = True; flush(L)
            if f and (pf is None or not pf): bk.reset(); st['resets'] += 1
            bk.upd(Bs[i], Ps[i], As[i]); pL = L; pf = f; pE = Es[i]
        if stop: break
        if verbose and st['rows'] // 5_000_000 != done // 5_000_000: print(f"   replayed {st['rows']:,} rows | {time.time()-t0:.0f}s | messages {st['messages']:,} | trades evaluated {S['j']:,}")
        done = st['rows']
    if started and not stop:
        end_msg(pL, pE); st['trades_after_last_message'] = n - S['j']; flush(2 ** 62)
    asarr = lambda a: np.frombuffer(a, np.int64).copy() if len(a) else np.zeros(0, np.int64)
    return {'stats': st, 'cls': cls, 'dist': dist, 'Db': Dbq, 'Di': Diq, 'Cp': Cpq, 'gsz': gsz, 'gvol': gvol, 'post_ok': post_ok, 'age': age, 'touchq': touchq, 'ML': asarr(ML), 'ME': asarr(ME), 'MB': asarr(MB), 'MA': asarr(MA), 'trades_evaluated': S['j'], 'final_levels': (len(bk.bids), len(bk.asks)), 'seconds': time.time() - t0}

In [ ]:
S25_STRIDE = 40; t0 = time.time(); SAMP, N_S25 = load_s25_best(S25_STRIDE); tick_('load_snapshot25_best_sample', t0); print(f'book_snapshot_25 rows: {N_S25:,} | every {S25_STRIDE}-th row sampled -> {len(SAMP):,} distinct local_timestamps')
t0 = time.time(); PS = run_pass(samp=SAMP); tick_('replay_walk_the_book', t0)
st = PS['stats']; print(st); print('final levels (bids, asks):', PS['final_levels'], '| trades evaluated:', PS['trades_evaluated'], 'of', NT)
cls = PS['cls']; MLn, MEn, MBn, MAn = PS['ML'], PS['ME'], PS['MB'], PS['MA']

[load_snapshot25_best_sample] 6.0 s | peak memory so far 1.33 GB
book_snapshot_25 rows: 790,101 | every 40-th row sampled -> 19,753 distinct local_timestamps
   replayed 5,457,704 rows | 20s | messages 250,025 | trades evaluated 362,871
   replayed 10,247,801 rows | 37s | messages 317,006 | trades evaluated 914,602
   replayed 15,030,188 rows | 53s | messages 463,980 | trades evaluated 1,282,071
   replayed 20,497,640 rows | 71s | messages 605,190 | trades evaluated 1,638,513
   replayed 25,286,884 rows | 89s | messages 690,822 | trades evaluated 1,998,924
   replayed 30,074,828 rows | 105s | messages 817,202 | trades evaluated 2,320,438
[replay_walk_the_book] 108.9 s | peak memory so far 2.99 GB
{'rows': 31005311, 'skipped_pre': 0, 'messages': 861138, 'resets': 1, 'crossed': 0, 'locked': 0, 'snap_checked': 19753, 'snap_best_equal': 19753, 'trades_after_last_message': 20, 'groups': 500059}
final levels (bids, asks): (26337, 31048) | trades evaluated: 2401094 of 2401094


## 5 · Results: how many trades are consistent with consuming the displayed book?

In [ ]:
valid = (cls <= 7); NV = int(valid.sum()); dist = PS['dist']; gsz = PS['gsz']; Q = valid & (dist >= 2); NQ = int(Q.sum())
def shares(mask):
    c = np.bincount(cls[mask], minlength=10)[:8]; return c, c / max(int(c.sum()), 1)
rows = {}
for nm, m in (('all', valid), ('sell_taker', valid & (tside == 1)), ('buy_taker', valid & (tside == 0)), ('deep_Q_(>=2_ticks_deeper)', Q), ('not_deep', valid & ~Q)):
    c, s_ = shares(m); rows[nm + '_count'] = c; rows[nm + '_share'] = s_
CLSTAB = pd.DataFrame(rows, index=[CLS_NAMES[i] for i in range(8)]); display(CLSTAB.style.format({c: '{:.3%}' for c in CLSTAB.columns if c.endswith('share')}))
W = float(CLSTAB['all_share'].iloc[0]); I_ = float(CLSTAB['all_share'].iloc[5] + CLSTAB['all_share'].iloc[6]); Qc = float(CLSTAB['deep_Q_(>=2_ticks_deeper)_share'].iloc[0])
Q_NOT_DISPLAYED = float(CLSTAB['deep_Q_(>=2_ticks_deeper)_share'].iloc[4]); print(f'usable trades {NV:,} | W (consistent, all) = {W:.2%} | I (better than touch / through opposite) = {I_:.2%} | deep trades Q = {NQ:,} ({NQ/max(NV,1):.2%}) | Qc (deep trades that are consistent) = {Qc:.2%}')
bins = [(1, 1, '1 trade'), (2, 3, '2-3 trades'), (4, 10, '4-10 trades'), (11, 10 ** 9, '>10 trades')]; BR = []
for lo_, hi_, lab in bins:
    m = valid & (gsz >= lo_) & (gsz <= hi_); n_ = int(m.sum()); r = {'burst_size_(same_side_trades_in_interval)': lab, 'trades': n_, 'share_of_usable': sh(n_, NV), 'P(deep)': sh(int((Q & m).sum()), n_)}
    for k_, nm in ((0, 'consistent'), (1, 'reach_violated'), (2, 'exhaustion_violated'), (3, 'both'), (4, 'not_displayed')): r[nm] = sh(int(((cls == k_) & m).sum()), n_)
    r['better_than_touch_or_through'] = sh(int((((cls == 5) | (cls == 6)) & m).sum()), n_); BR.append(r)
BURST = pd.DataFrame(BR); display(BURST.style.format({c: '{:.2%}' for c in BURST.columns if c not in ('burst_size_(same_side_trades_in_interval)', 'trades')}))

,all_count,all_share,sell_taker_count,sell_taker_share,buy_taker_count,buy_taker_share,deep_Q_(>=2_ticks_deeper)_count,deep_Q_(>=2_ticks_deeper)_share,not_deep_count,not_deep_share
consistent,1380901,57.512%,713464,56.690%,667437,58.418%,200545,18.962%,1180356,87.862%
reach_violated(deeper_than_volume_explains),677779,28.228%,363852,28.911%,313927,27.477%,663006,62.688%,14773,1.100%
exhaustion_violated(more_than_displayed),115733,4.820%,61835,4.913%,53898,4.717%,76067,7.192%,39666,2.953%
both_violated,867,0.036%,438,0.035%,429,0.038%,857,0.081%,10,0.001%
price_not_displayed_level,118915,4.953%,69604,5.531%,49311,4.316%,117152,11.077%,1763,0.131%
better_than_touch_inside_spread,16500,0.687%,7663,0.609%,8837,0.773%,0,0.000%,16500,1.228%
at_or_through_opposite_touch,90359,3.763%,41675,3.311%,48684,4.261%,0,0.000%,90359,6.726%
beyond_walk_cap,0,0.000%,0,0.000%,0,0.000%,0,0.000%,0,0.000%


usable trades 2,401,054 | W (consistent, all) = 57.51% | I (better than touch / through opposite) = 4.45% | deep trades Q = 1,057,627 (44.05%) | Qc (deep trades that are consistent) = 18.96%


,burst_size_(same_side_trades_in_interval),trades,share_of_usable,P(deep),consistent,reach_violated,exhaustion_violated,both,not_displayed,better_than_touch_or_through
0,1 trade,269365,11.22%,0.68%,94.08%,0.67%,0.01%,0.00%,0.09%,5.15%
1,2-3 trades,252686,10.52%,6.03%,85.28%,4.54%,0.40%,0.00%,0.71%,9.06%
2,4-10 trades,422584,17.60%,33.00%,65.75%,20.74%,3.27%,0.04%,4.91%,5.30%
3,>10 trades,1456419,60.66%,61.87%,43.54%,39.61%,6.93%,0.05%,6.60%,3.27%


## 6 · How large are the violations, and what does the book do afterwards?

* **Reach violated** means the price went deeper than the traded volume explains. The *shortfall* is how much displayed quantity ahead of the print was *not* consumed by prior trades (cancelled, moved, or consumed by trades of a different feed order).
* **Exhaustion violated** means more volume traded than the book displayed at/above that level. The *excess* is how much.
* **Post-interval touch:** after the next message, did the touch on the hit side move to (or through) the deepest price traded? If yes, the liquidity ahead of that print really is gone; if no, better-priced liquidity is still (or again) displayed.

In [ ]:
amt = tam; Db, Di, Cp = PS['Db'], PS['Di'], PS['Cp']; pok = PS['post_ok']
def qs(a): return [float(x) for x in np.percentile(a, [50, 75, 90, 99])] if len(a) else None
m1 = (cls == 1) | (cls == 3); m2 = (cls == 2) | (cls == 3)
short = (Db - Cp)[m1]; excess = (Cp + amt - Di)[m2]
MAG = {'reach_violated_trades': int(m1.sum()), 'shortfall_BTC_quantiles(50,75,90,99)': qs(short), 'shortfall_as_share_of_better_depth_quantiles(50,75,90,99)': qs((short / np.maximum(Db[m1], 1e-12))) if m1.any() else None,
       'exhaustion_violated_trades': int(m2.sum()), 'excess_BTC_quantiles(50,75,90,99)': qs(excess), 'median_trade_amount_BTC': float(np.median(tam)),
       'interval_volume_over_touch_qty_quantiles(50,75,90,99)_for_deep_trades': qs((PS['gvol'] / np.maximum(PS['touchq'], 1e-12))[Q & (PS['touchq'] > 0)])}
POST = {}
for k_, nm in ((0, 'consistent'), (1, 'reach_violated'), (2, 'exhaustion_violated'), (3, 'both'), (4, 'not_displayed')):
    m = (cls == k_) & (pok >= 0); POST[nm] = {'trades_with_post_state': int(m.sum()), 'touch_moved_to_or_through_deepest_print': sh(int((pok[m] == 1).sum()), int(m.sum()))}
m = Q & (pok >= 0); POST['all_deep_Q'] = {'trades_with_post_state': int(m.sum()), 'touch_moved_to_or_through_deepest_print': sh(int((pok[m] == 1).sum()), int(m.sum()))}
vol_w = {CLS_NAMES[k_]: float(tam[(cls == k_)].sum() / max(tam[valid].sum(), 1e-12)) for k_ in range(8)}
print(json.dumps({'magnitudes': MAG, 'post_interval_touch': POST, 'volume_weighted_class_shares': vol_w}, indent=1, default=jsonable))
# time of day and volatility (consistent share)
hour = np.clip((tL - DAY0_US) // 3_600_000_000, 0, 23).astype(int); HOUR = pd.DataFrame({'h': hour[valid], 'c': (cls[valid] == 0)}).groupby('h').c.agg(['mean', 'size']).rename(columns={'mean': 'consistent_share', 'size': 'trades'})
mn = np.clip((tL - DAY0_US) // 60_000_000, 0, 1439).astype(int); g = pd.DataFrame({'m': mn, 'p': tpx}).groupby('m').p; rng_bps = ((g.max() - g.min()) / g.mean() * 1e4).rename('rng_bps')
dfv = pd.DataFrame({'m': mn[valid], 'c': (cls[valid] == 0)}).join(rng_bps, on='m'); dfv['vol_quintile'] = pd.qcut(dfv.rng_bps.rank(method='first'), 5, labels=['Q1 calmest', 'Q2', 'Q3', 'Q4', 'Q5 most volatile']); VOL = dfv.groupby('vol_quintile', observed=True).agg(consistent_share=('c', 'mean'), trades=('c', 'size'))
display(HOUR.style.format({'consistent_share': '{:.2%}'})); display(VOL.style.format({'consistent_share': '{:.2%}'}))
REPORT['classes'] = {'table': CLSTAB.to_dict(), 'W_consistent_all': W, 'I_better_or_through': I_, 'Qc_deep_consistent': Qc, 'usable_trades': NV, 'deep_trades': NQ}; REPORT['burst_table'] = BURST.to_dict('records'); REPORT['magnitudes'] = MAG; REPORT['post_interval_touch'] = POST; REPORT['volume_weighted'] = vol_w; REPORT['time_of_day'] = HOUR.reset_index().to_dict('records'); REPORT['volatility'] = VOL.reset_index().to_dict('records')

{
 "magnitudes": {
  "reach_violated_trades": 678646,
  "shortfall_BTC_quantiles(50,75,90,99)": [
   0.57382,
   1.4687800000000002,
   3.435560000000001,
   12.40918050000014
  ],
  "shortfall_as_share_of_better_depth_quantiles(50,75,90,99)": [
   0.5808226349189587,
   0.8098958450823976,
   0.958685698683099,
   1.0
  ],
  "exhaustion_violated_trades": 116600,
  "excess_BTC_quantiles(50,75,90,99)": [
   0.11156,
   0.33344999999999825,
   0.7835680000000017,
   4.94406249999999
  ],
  "median_trade_amount_BTC": 0.00267,
  "interval_volume_over_touch_qty_quantiles(50,75,90,99)_for_deep_trades": [
   2.827869051374767,
   13.590822179732312,
   128.44519015659955,
   19268.0
  ]
 },
 "post_interval_touch": {
  "consistent": {
   "trades_with_post_state": 1380881,
   "touch_moved_to_or_through_deepest_print": 0.9217159190400911
  },
  "reach_violated": {
   "trades_with_post_state": 677779,
   "touch_moved_to_or_through_deepest_print": 0.8109796851186006
  },
  "exhaustion_violated": {

,consistent_share,trades
h,,
0,60.47%,63852
1,61.10%,69967
2,62.43%,66991
3,69.89%,34307
4,72.30%,30988
5,68.27%,41352
6,66.41%,56142
7,51.73%,340170
8,57.03%,228598


,consistent_share,trades
vol_quintile,,
Q1 calmest,69.23%,480211
Q2,61.70%,480211
Q3,57.10%,480210
Q4,52.77%,480211
Q5 most volatile,46.76%,480211


## 7 · Execution tests (E) and input-data integrity tests (D)

In [ ]:
import random
TESTS = {'E': [], 'D': []}
def tcase(group, name, fn):
    try: ok, msg = fn()
    except Exception as e: ok, msg = False, f'EXC {type(e).__name__}: {str(e)[:200]}'
    TESTS[group].append((name, bool(ok), msg)); print(('PASS ' if ok else 'FAIL ') + name + ' | ' + str(msg))
def brute(dct, touch, dirn, opp, prices, amts, cap=WALK_CAP):
    # independent re-implementation: no incremental walk; depth better than p is summed directly from the dictionary
    out = []; C = 0.0
    for p, a in zip(prices, amts):
        d = (p - touch) * dirn
        if d < 0: out.append(6 if ((dirn < 0 and p >= opp) or (dirn > 0 and p <= opp)) else 5); continue
        if d > cap: out.append(7); continue
        if p not in dct: out.append(4); continue
        db = sum(q for pk, q in dct.items() if (pk - p) * dirn < 0); di = db + dct[p]
        reach = C >= db - TOL_ABS - TOL_REL * db; exh = (C + a) <= di + TOL_ABS + TOL_REL * di
        out.append(0 if (reach and exh) else 1 if (not reach and exh) else 2 if (reach and not exh) else 3); C += a
    return out
def e01():
    now = {str(p.relative_to(BASE)): (p.stat().st_size, p.stat().st_mtime_ns) for p in BASE.rglob('*') if p.is_file() and not str(p.relative_to(BASE)).startswith(STAGE.name)}
    bad = [k for k in PIN0 if now.get(k) != PIN0[k]]; new = [k for k in now if k not in PIN0]; return (not bad and not new), (f'{len(PIN0)} pre-existing files (incl. notebooks 07 and 08 outputs) unchanged; nothing new outside staging' if not bad and not new else f'CHANGED {bad[:3]} NEW {new[:3]}')
def e02():
    sell = {100: 1.0, 99: 2.0, 98: 1.0, 96: 5.0}; buy = {101: 1.0, 102: 2.0, 103: 1.0, 105: 5.0}
    cases = [(sell, 100, -1, 103, [100, 100], [0.4, 0.4], [0, 0]), (sell, 100, -1, 103, [100, 100], [1.0, 0.1], [0, 2]), (sell, 100, -1, 103, [99], [0.3], [1]), (sell, 100, -1, 103, [100, 99], [1.0, 0.5], [0, 0]), (sell, 100, -1, 103, [97], [0.1], [4]),
             (sell, 100, -1, 103, [101], [0.1], [5]), (sell, 100, -1, 101, [101], [0.1], [6]), (sell, 100, -1, 103, [100, 99, 98], [1.0, 2.0, 0.5], [0, 0, 0]), (sell, 100, -1, 103, [100 - 40000], [0.1], [7]), (sell, 100, -1, 103, [98, 98], [0.2, 0.2], [1, 1]), (sell, 100, -1, 103, [98], [5.0], [3]),
             (buy, 101, 1, 100, [101, 101], [0.4, 0.7], [0, 2]), (buy, 101, 1, 100, [102], [0.3], [1]), (buy, 101, 1, 100, [101, 102, 103], [1.0, 2.0, 0.4], [0, 0, 0]), (buy, 101, 1, 100, [104], [0.1], [4]), (buy, 101, 1, 98, [100], [0.1], [5]), (buy, 101, 1, 100, [100], [0.1], [6])]
    bad = []
    for i, (d, t, dr, o, p, a, exp) in enumerate(cases):
        got = walk_classify(d, t, dr, o, p, a)[0]
        if got != exp: bad.append((i, exp, got))
    return not bad, f'{len(cases)} hand-made interval cases (sells and buys; exact exhaustion, both tests violated, refill, skipped depth, undisplayed level, improved price, through opposite, walk cap) all classified as expected' if not bad else f'WRONG {bad[:3]}'
def e03():
    rnd = random.Random(12345); bad = 0; N = 4000
    for _ in range(N):
        dr = rnd.choice([-1, 1]); touch = 1000; dct = {}
        for k in range(0, 80):
            if k == 0 or rnd.random() < 0.7: dct[touch + dr * k] = round(rnd.uniform(0.01, 2.0), 3)
        opp = touch - dr * rnd.randint(1, 4); n = rnd.randint(1, 8); pr = [touch + dr * rnd.randint(-3, 60) for _ in range(n)]; am = [round(rnd.uniform(0.001, 1.5), 3) for _ in range(n)]
        if walk_classify(dct, touch, dr, opp, pr, am)[0] != brute(dct, touch, dr, opp, pr, am): bad += 1
    return bad == 0, f'walk_classify == independent brute-force implementation on {N:,} random books/trade sequences ({bad} disagreements)'
def e04():
    m_ok = st['messages'] == SRC_REPORT['coverage']['messages']; r_ok = st['rows'] == SRC_REPORT['book_scan']['rows']; c_ok = st['crossed'] == SRC_REPORT['crossed']['unpruned']['crossed_messages'] and st['locked'] == SRC_REPORT['crossed']['unpruned']['locked_messages']
    mm = re.search(r'unpruned: messages [\d,]+ .*?final levels \((\d+), (\d+)\)', SRC_SHARE); f_ok = bool(mm) and PS['final_levels'] == (int(mm.group(1)), int(mm.group(2)))
    return m_ok and r_ok and c_ok and f_ok, f"replay == notebook 07 record: messages {st['messages']:,} {m_ok}, rows {r_ok}, crossed/locked {c_ok}, final levels {PS['final_levels']} {f_ok}"
def e05():
    a = run_pass(max_rows=300_000, verbose=False); b = run_pass(max_rows=300_000, verbose=False); dig = lambda r: hashlib.sha256(r['cls'].tobytes() + r['dist'].tobytes() + r['Db'].tobytes() + r['Cp'].tobytes() + r['post_ok'].tobytes() + r['ML'].tobytes()).hexdigest()[:16]
    return dig(a) == dig(b) and a['stats']['messages'] == b['stats']['messages'], f"two 300k-row runs identical (digest {dig(a)}, {a['stats']['messages']:,} messages, {a['trades_evaluated']:,} trades evaluated)"
def e06():
    m = PS['age'] >= 0; a_ok = bool(np.all(PS['age'][m] > 0)); idx = np.searchsorted(MLn, tL, side='left') - 1; c_ok = bool(np.all(MLn[idx[idx >= 0]] < tL[idx >= 0])); return a_ok and c_ok, 'every trade was judged against a book interval that started strictly BEFORE it (age > 0; state timestamp < trade timestamp) - no look-ahead'
def e07():
    tot = int(np.bincount(cls, minlength=10).sum()); sums = {c: float(CLSTAB[c].sum()) for c in CLSTAB.columns if c.endswith('share')}; ok = tot == NT and all(abs(v_ - 1.0) < 1e-9 for v_ in sums.values()); return ok, f'every one of {NT:,} trades has exactly one class; shares sum to 1: {sums}'
def e08():
    if not DIAG08.is_file(): return True, 'notebook 08 report not found -> cross-notebook reproduction skipped'
    rep = json.loads(DIAG08.read_text()); q08 = rep['categories']['Q_count']; thr08 = rep['categories']['table']['all_trades_count']['through_opposite']; ins08 = rep['categories']['table']['all_trades_count']['inside_spread']
    okq = int(NQ) == int(q08); okt = int((cls == 6).sum()) == int(thr08); oki = int((cls == 5).sum()) == int(ins08); return okq and okt and oki, f'pre-trade categories reproduce notebook 08: deep {NQ:,} vs {q08:,} {okq}; through-opposite {int((cls==6).sum()):,} vs {thr08:,} {okt}; inside-spread {int((cls==5).sum()):,} vs {ins08:,} {oki}'
def e09():
    if not HAVE_S25: return True, 'book_snapshot_25 not present -> best bid/ask re-check skipped'
    n_ = st['snap_checked']; r = sh(st['snap_best_equal'], n_); need = min(5000, len(SAMP) // 2); return n_ >= need and r >= 0.999, f'best bid & ask equal to book_snapshot_25 at {n_:,} aligned checkpoints: {r:.4%} (need >= 99.9% on >= {need:,}; same-provider consistency only)'
for nm, fn in (('E01 nothing pre-existing modified (read-only)', e01), ('E02 walk-the-book correct on hand-made cases', e02), ('E03 walk-the-book == brute-force implementation', e03), ('E04 replay reproduces notebook 07 numbers', e04), ('E05 replay deterministic', e05), ('E06 no look-ahead in the book state', e06), ('E07 classes partition all trades', e07), ('E08 pre-trade categories reproduce notebook 08', e08), ('E09 best bid/ask re-check vs book_snapshot_25', e09)): tcase('E', nm, fn)
def d01(): bad = [k for k, v_ in SHA_CHECK.items() if not v_['equal']]; return bool(SHA_CHECK) and not bad, ('SHA-256 of every saved raw file equals the value recorded by notebook 07' if not bad else f'MISMATCH {bad}')
def d02(): a = NT == SRC_REPORT['trades_scan']['rows']; b = st['rows'] == SRC_REPORT['book_scan']['rows']; return a and b, f"row counts equal notebook 07: trades {NT:,} {a}, book {st['rows']:,} {b}"
def d03(): return GRID['book_first_block_off_grid_share'] <= 1e-4 and GRID['trades_off_grid_share'] <= 1e-4, f'prices lie on the {TICK} tick grid: {GRID}'
def d04(): ok = bool(np.all(tpx > 0)) and bool(np.all(tam > 0)) and sh(int((tside == 2).sum()), NT) <= 0.01 and bool(np.all(np.isfinite(tpx))); return ok, f'trade price>0, amount>0, finite; unknown-side share {sh(int((tside == 2).sum()), NT):.4%}'
def d05(): dif = np.diff(tid); return bool(np.all(dif > 0)), f'trade IDs strictly increasing; gaps(>1): {int((dif > 1).sum())}'
def d06(): inside = float(np.mean((tL >= DAY0_US) & (tL < DAY1_US))); return inside >= 0.999, f'trade local_timestamp inside {DAY} UTC: {inside:.4%}'
def d07(): return bool(np.all(np.diff(MLn) > 0)) and len(MLn) > 1000, f'book message local timestamps strictly increasing ({len(MLn):,} messages)'
for nm, fn in (('D01 saved raw files match notebook 07 SHA-256', d01), ('D02 row counts match notebook 07', d02), ('D03 prices on tick grid', d03), ('D04 trade values sane', d04), ('D05 trade IDs increasing', d05), ('D06 trades inside the day', d06), ('D07 book messages strictly ordered', d07)): tcase('D', nm, fn)
E_OK = all(t[1] for t in TESTS['E']); D_OK = all(t[1] for t in TESTS['D'])
print(f"\nE (execution): {sum(t[1] for t in TESTS['E'])}/{len(TESTS['E'])} | D (input integrity): {sum(t[1] for t in TESTS['D'])}/{len(TESTS['D'])}")

PASS E01 nothing pre-existing modified (read-only) | 768 pre-existing files (incl. notebooks 07 and 08 outputs) unchanged; nothing new outside staging
PASS E02 walk-the-book correct on hand-made cases | 17 hand-made interval cases (sells and buys; exact exhaustion, both tests violated, refill, skipped depth, undisplayed level, improved price, through opposite, walk cap) all classified as expected
PASS E03 walk-the-book == brute-force implementation | walk_classify == independent brute-force implementation on 4,000 random books/trade sequences (0 disagreements)
PASS E04 replay reproduces notebook 07 numbers | replay == notebook 07 record: messages 861,138 True, rows True, crossed/locked True, final levels (26337, 31048) True
PASS E05 replay deterministic | two 300k-row runs identical (digest 67f87da7ad8de750, 11,959 messages, 19,373 trades evaluated)
PASS E06 no look-ahead in the book state | every trade was judged against a book interval that started strictly BEFORE it (age > 0; state 

## 8 · Recommendation (rule frozen in the introduction) and the FINAL WALK-THE-BOOK SHARE BLOCK

In [ ]:
def qlabel(x): return 'n/a' if x != x else 'LARGELY consistent with consuming the displayed book' if x >= 0.60 else 'PARTLY consistent' if x >= 0.30 else 'MOSTLY NOT consistent'
if not (E_OK and D_OK): REC = 'HOLD - an execution or integrity test failed; fix that first.'
elif W >= 0.70 and I_ <= 0.06: REC = 'PROCEED - ask for approval of the remaining free days (BTCUSDT and ETHUSDT, 1st of each month in the study window). Use trades only as per-100ms-interval aggregates.'
elif W >= 0.40: REC = 'PROCEED FOR BOOK DATA ONLY - ask for approval of the remaining free days for the order-book stream, but do NOT match trades price-by-price to the book; use trades only as per-100ms-interval aggregates (signed volume, count, range).'
else: REC = 'HOLD - fewer than 40% of trades are consistent with the displayed book; do not download more until this is understood or raised with Tardis.'
print(REC)
def pct(x, d=2): return 'nan' if x != x else f'{x:.{d}%}'
unc = ['One symbol and one day.', 'A violation of either test is not a data error: the book is a 100 ms net-change feed, so cancellations, new orders and refills inside an interval are invisible by construction. The tests measure how much of the tape can be tied to the displayed book, nothing more.',
       'Trade order inside an interval follows the file (local arrival) order; trades of one interval that arrive out of true execution order would blur the reach/exhaustion tests.',
       'Book and trades come from the same provider; "consistent" is not independent validation. L2 data has no order identity: nothing here identifies orders or traders and nothing here detects manipulation.',
       'The 60% / 30% / 70% / 40% / 6% cut-offs are judgement calls fixed in advance, not statistical tests.', 'Publication of results is not addressed in the Tardis terms: ask Tardis in writing. Raw files, per-trade tables derived from them, and trained models must not be given to third parties.']
L = []; P = L.append
P('=' * 100); P('FINAL WALK-THE-BOOK SHARE BLOCK - Tardis.dev Binance Spot BTCUSDT 2024-05-01 (read-only follow-up to notebooks 07 and 08)'); P('=' * 100)
P(f'run: {RUN_STARTED} | source (read-only): {SRC} | output: {OUT_FINAL} | network used: NO | downloads: NONE')
P('Question: at 100 ms resolution, are trades consistent with the displayed full-depth book being consumed? Reach test + exhaustion test per book interval and taker side. Not a validation of anything; no manipulation claims.')
P('\n[A] INPUTS'); [P(f"  {v_['file']}: sha256 equal to notebook 07 manifest: {v_['equal']}") for v_ in SHA_CHECK.values()]; P(f'  trades {NT:,} | book rows {st["rows"]:,} | tick {TICK} | usable trades {NV:,} | (interval,side) groups {st["groups"]:,}')
P('\n[B] REFERENCE REPLAY'); P(f"  messages {st['messages']:,} | crossed {st['crossed']} | locked {st['locked']} | resets {st['resets']} | final levels {PS['final_levels']} | best bid/ask equal to book_snapshot_25 at {st['snap_checked']:,} aligned checkpoints: {pct(sh(st['snap_best_equal'], st['snap_checked']), 4)} (same provider: consistency only)")
P('\n[C] TRADE CLASSES'); P(CLSTAB.assign(**{c: CLSTAB[c].map('{:.3%}'.format) for c in CLSTAB.columns if c.endswith('share')}).to_string())
P(f'  W (consistent, all usable) = {pct(W)} | I (better than touch or through opposite) = {pct(I_)} | deep trades Q = {NQ:,} ({pct(sh(NQ, NV))}) | Qc (deep and consistent) = {pct(Qc)} -> deep trades are {qlabel(Qc)} | deep trades at a price that is not a displayed level: {pct(Q_NOT_DISPLAYED)}')
P('\n[D] BY BURST SIZE (same-side trades in one book interval)'); P(BURST.assign(**{c: BURST[c].map('{:.2%}'.format) for c in BURST.columns if c not in ('burst_size_(same_side_trades_in_interval)', 'trades')}).to_string(index=False))
P('\n[E] SIZE OF VIOLATIONS'); P('  ' + json.dumps(MAG, default=jsonable))
P('\n[F] BOOK AFTER THE INTERVAL (touch moved to/through the deepest print?)'); P('  ' + json.dumps(POST, default=jsonable)); P('  volume-weighted class shares: ' + json.dumps(vol_w, default=jsonable))
P('\n[G] TIME OF DAY / VOLATILITY (share consistent)'); P(HOUR.assign(consistent_share=HOUR.consistent_share.map('{:.1%}'.format)).T.to_string()); P(VOL.assign(consistent_share=VOL.consistent_share.map('{:.1%}'.format)).to_string())
P('\n' + '-' * 100); P('[E] NOTEBOOK EXECUTION TESTS'); [P(f"  {'PASS' if ok else 'FAIL'} {n} | {m}") for n, ok, m in TESTS['E']]; P(f"  => E: {sum(t[1] for t in TESTS['E'])}/{len(TESTS['E'])}")
P('[D] INPUT-DATA INTEGRITY TESTS'); [P(f"  {'PASS' if ok else 'FAIL'} {n} | {m}") for n, ok, m in TESTS['D']]; P(f"  => D: {sum(t[1] for t in TESTS['D'])}/{len(TESTS['D'])}")
P('  Passing E and D means the code and the files are sound; it does NOT validate the book or the trades.')
REQ = {'stage_seconds': STAGES, 'peak_memory_gb': round(peak_mem_gb(), 2), 'replay_seconds': round(PS['seconds'], 1)}; P('\n[H] PROCESSING'); P('  ' + json.dumps(REQ, default=jsonable))
P('\nRECOMMENDATION (rule-based, frozen in advance; nothing is downloaded without your approval):'); P('  ' + REC)
P('\nUNCERTAINTIES'); [P('  - ' + u) for u in unc]
P('\nNothing was downloaded. No models, graphs or synthetic scenarios were created. Notebooks 07 and 08 outputs and all earlier artifacts were not modified. WAITING FOR YOUR APPROVAL before the next stage.'); P('=' * 100)
SHARE = '\n'.join(L)
STAGE.mkdir(exist_ok=True); (STAGE / 'reports').mkdir(exist_ok=True); REPORT['tests'] = TESTS; REPORT['recommendation'] = REC; REPORT['uncertainties'] = unc; REPORT['finished'] = UTC(); REPORT['processing'] = REQ
dump_json(REPORT, STAGE / 'reports' / 'walk_the_book_report.json'); (STAGE / 'reports' / 'share_block.txt').write_text(SHARE, encoding='utf-8')
CLSTAB.to_csv(STAGE / 'reports' / 'class_table.csv'); BURST.to_csv(STAGE / 'reports' / 'burst_table.csv', index=False); HOUR.to_csv(STAGE / 'reports' / 'hour_table.csv'); VOL.to_csv(STAGE / 'reports' / 'volatility_table.csv')
pd.DataFrame({'trade_id': tid, 'side_code(0buy,1sell,2other)': tside, 'class_code': cls, 'ticks_deeper_than_touch': np.clip(dist, -10 ** 6, 10 ** 6).astype(np.int32), 'same_side_trades_in_interval': gsz, 'touch_moved_to_or_through_deepest_print': pok}).to_parquet(STAGE / 'reports' / 'per_trade_walk_class.parquet', index=False)
MAN2 = {str(p.relative_to(STAGE)): sha256_file(p) for p in sorted(STAGE.rglob('*')) if p.is_file()}; dump_json(MAN2, STAGE / 'manifest.json')
okm = all(sha256_file(STAGE / k) == v_ for k, v_ in MAN2.items()); all_ok = E_OK and D_OK and okm
if all_ok: os.rename(STAGE, OUT_FINAL); print('\nSAVED ->', OUT_FINAL, '(results; the per-trade table is derived from licensed data - keep private)')
else: print('\n!! E or D tests FAILED (or manifest mismatch) -> staging kept at', STAGE, '(not renamed). Share this block anyway.')
print(SHARE); print(f'manifest verified: {okm} ({len(MAN2)} files)')

PROCEED FOR BOOK DATA ONLY - ask for approval of the remaining free days for the order-book stream, but do NOT match trades price-by-price to the book; use trades only as per-100ms-interval aggregates (signed volume, count, range).

SAVED -> /content/drive/MyDrive/research datasets/TGNN_Tardis_WalkBook_BTCUSDT_2024-05-01_v1 (results; the per-trade table is derived from licensed data - keep private)
FINAL WALK-THE-BOOK SHARE BLOCK - Tardis.dev Binance Spot BTCUSDT 2024-05-01 (read-only follow-up to notebooks 07 and 08)
run: 2026-10-12T07:14:00.325954+00:00 | source (read-only): /content/drive/MyDrive/research datasets/TGNN_Tardis_Binance_Spot_BTCUSDT_2024-05-01_v1 | output: /content/drive/MyDrive/research datasets/TGNN_Tardis_WalkBook_BTCUSDT_2024-05-01_v1 | network used: NO | downloads: NONE
Question: at 100 ms resolution, are trades consistent with the displayed full-depth book being consumed? Reach test + exhaustion test per book interval and taker side. Not a validation of anything;